# When Qini Rewards the Wrong Uplift Model

This notebook follows the evaluation article. First run `python reproduce.py --output regenerated`. This notebook then analyzes those generated results; it does not refit models. It checks defining evaluator formulas, paired uncertainty and validation-selected tuning. All six boosted learners remain in every model comparison. Synthetic truth is a diagnostic, never a nuisance-model input.

Assumptions: consistency, no relevant interference, exchangeability given X, and overlap. Figures and rankings do not establish those assumptions in real observational data.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
RUN_ROOT=Path('regenerated')  # Change to Path('regenerated') after a full rerun.
root=RUN_ROOT/'data/selection'
results=pd.read_csv(root/'comparison_calibrated.csv')
primary=results.query("scenario=='selection_high' and seed==42").set_index('Model')
names=['S','T','X','DA','DR','DML']
keys=['TrueCATE','linear/DR']+['xgb/'+m for m in names]
assert set(keys).issubset(primary.index)
print('Saved experiment: N=1,000,000; main holdout=300,000; seed42')
calibration=json.loads((root/"calibration.json").read_text())
np.testing.assert_allclose(primary.loc["TrueCATE",["Qini_calibrated","IPW_calibrated","DR_estimated"]].to_numpy(float),calibration["reference_DR"],atol=1e-14)
print("Reference scores match; calibration constants frozen across all comparisons.")

## The opening reversal

Compare within each metric column. Linear DR was chosen as an illustrative diagnostic after inspecting results; this is not a prespecified universal learner claim.

In [ ]:
display(primary.loc[['TrueCATE','linear/DR'],['Qini_calibrated','Oracle_obs']])
expected=pd.read_csv(RUN_ROOT/'data/support/expected_counts.csv').set_index('Model')
print('Raw expected-count Qini diagnostic (audit only; not article Qini/share):')
display(expected.loc[['TrueCATE','linear/DR'],['ExpectedObservational','OracleGain']])

## Assignment and ordinary evaluation

The population treatment share is fixed near50%, while assignment varies with covariates. These figures introduce the same S ranking used in the standalone article example.

In [ ]:
summary=pd.read_csv(root/'summaries.csv')
display(summary.query("scenario=='selection_high' and seed==42")[[ 'treatment_rate','ATT','ATC','min_propensity','max_propensity']])
for f in ['01_meet_the_data.png','03_uplift_bars.png','04_qini_for_s.png']:
    display(Image(filename=str(RUN_ROOT/'figures/narrative'/f)))

## Reconstruct IPW and DR without synthetic truth

The audit sample contains1,000 observed evaluation rows and their out-of-fold nuisance predictions. Recompute each pseudo-outcome independently and compare with the saved evaluator outputs. These sampled rows do not reproduce the full300k area on their own.

In [ ]:
a=pd.read_csv(RUN_ROOT/'data/support/formula_sample.csv')
t=a.TreatmentGroup.to_numpy();y=a.MadeOrder.to_numpy();e=a.e_hat.to_numpy()
mu0=a.mu0_hat.to_numpy();mu1=a.mu1_hat.to_numpy()
ipw=t*y/e-(1-t)*y/(1-e)
dr=mu1-mu0+t*(y-mu1)/e-(1-t)*(y-mu0)/(1-e)
np.testing.assert_allclose(ipw,a.IPW_saved,atol=1e-12)
np.testing.assert_allclose(dr,a.DR_saved,atol=1e-12)
print('Independent IPW/DR formula audit passed on1,000 observed rows.')
display(primary.loc[keys,['Qini_calibrated','IPW_calibrated','DR_estimated','Oracle_obs']])
display(Image(filename=str(RUN_ROOT/'figures/narrative/06_all_learner_evaluations.png')))

## Conditional uncertainty and paired differences

1,000 identical row resamples across models. Ranking and nuisance fits remain fixed. These are not full training/procedure confidence intervals; comparisons are unadjusted for multiplicity.

Calibration constants are also held fixed. These intervals exclude uncertainty in estimating the reference calibration.

In [ ]:
for method in ['Qini_calibrated','IPW_calibrated','DR']:
    ci=pd.read_csv(root/'intervals'/f'{method}_intervals.csv').set_index('Model')
    draws=pd.read_csv(root/'intervals'/f'{method}_draws.csv')
    for key in keys:
        np.testing.assert_allclose(np.quantile(draws[key],[.025,.975]),ci.loc[key,['CI_low','CI_high']].to_numpy(float),atol=1e-12)
    print(method);display(ci.loc[keys])
pairs=pd.read_csv(root/'intervals/DR_pairs.csv')
paired=pairs[(pairs.A=='xgb/S') & pairs.B.isin(['xgb/'+m for m in names[1:]])]
for _,row in paired.iterrows():
    np.testing.assert_allclose(np.quantile(draws[row.A]-draws[row.B],[.025,.975]),[row.CI_low,row.CI_high],atol=1e-12)
display(paired)
display(Image(filename=str(RUN_ROOT/'figures/narrative/05_paired_confidence_intervals.png')))
display(Image(filename=str(RUN_ROOT/'figures/narrative/05_oracle_paired_comparison.png')))

## Assignment policies

Across-policy whiskers are three-seed ranges, not confidence intervals. Calibration constants are frozen at the seed-42 high-selectivity reference.

In [ ]:
fixed=results[results.Model=='TrueCATE'].copy()
for key in ['Qini_calibrated_error','IPW_calibrated_error','DR_error']:fixed[key+'_abs']=fixed[key].abs()
display(fixed.groupby('scenario')[[ 'Qini_calibrated_error','IPW_calibrated_error_abs','DR_error_abs']].mean())
display(Image(filename=str(RUN_ROOT/'figures/selection/03_evaluation_bias.png')))

## Tune against adjusted validation and test on fresh customers

All six learners receive15candidate configurations. Winners are frozen before the fresh300k test is generated. Both baseline and tuned fits use the same700k training rows and the same new test rows. Conditional paired intervals omit search and retraining uncertainty.

In [ ]:
tuning=RUN_ROOT/'data/tuning'
h=pd.read_csv(tuning/'search_history.csv');frozen=json.loads((tuning/'selection_frozen.json').read_text())
assert h.groupby('Learner').size().eq(15).all()
for learner,winner in frozen['selected'].items():
    selected=h[h.Learner==learner].sort_values(['ValidationDR','config_id'],ascending=[False,True]).iloc[0]
    assert int(selected.config_id)==winner
change=pd.read_csv(tuning/'before_after.csv');d=pd.read_csv(tuning/'dr_draws.csv')
for _,r in change.iterrows():
    np.testing.assert_allclose(np.quantile(d[r.Learner+'/After']-d[r.Learner+'/Before'],[.025,.975]),[r.CI_low,r.CI_high],atol=1e-12)
display(change)
for fig in ['01_before_after.png','02_tuning_intervals.png']:display(Image(filename=str(RUN_ROOT/'figures/tuning'/fig)))

## Optional full regeneration

Run `python reproduce.py --output regenerated` from this directory. This refits all experiments and reruns resampling in a new directory. Set RUN_ROOT above to that directory to analyze the fresh outputs. `article_example.py` separately runs the standalone S/IPW/DR example at N=1M. See README for environment and scope; the article tables are generated from full experiment results, not the toy audit sample.